# Body Performance Analytics — Part 2: ML Model Training & Evaluation
**Course**: Introduction to AI and ML  
**Dataset**: Body Performance (Kaggle)  
**Authors**: Team Project


In [2]:
#!/usr/bin/env python3
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, learning_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, r2_score,
                             mean_absolute_error, mean_squared_error,
                             roc_curve, auc, RocCurveDisplay)
import xgboost as xgb
import os, warnings, time
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)
os.makedirs('plots', exist_ok=True)
plot_num = [25]  # Continue numbering from Part 1
def save(fig, title):
    plot_num[0] += 1
    fname = f"plots/P{plot_num[0]:02d}_{title.replace(' ','_').replace('/','_')[:40]}.png"
    fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [Saved: {fname}]")
    return fname

## SECTION 0: DATA LOADING & FEATURE ENGINEERING


In [3]:
print("=" * 70)
print("  PART 2: ML MODEL TRAINING & EVALUATION")
print("=" * 70)

df = pd.read_csv('bodyPerformance_cleaned.csv')
print(f"\nLoaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

  PART 2: ML MODEL TRAINING & EVALUATION

Loaded: 13,272 rows, 12 columns


### Feature Engineering


In [4]:
# WHY: Gender is categorical. ML needs numeric. Binary encoding (M=1) is clean.
df['genderEncoded'] = (df['gender'] == 'M').astype(int)

# WHY: BMI captures weight-to-height ratio in a single metabolic indicator.
df['bmi'] = df['weightKg'] / (df['heightCm'] / 100) ** 2

# WHY: 40 sit-ups means different things for men vs women. Z-scoring within
# gender removes this confound so the model sees RELATIVE performance.
df['sitUpsCounts_zGender'] = df.groupby('gender')['sitUpsCounts'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# WHY: Raw grip force penalizes lighter people. Normalizing by lean body mass
# captures "pound-for-pound" strength — a better predictor of athletic class.
lean_mass = df['weightKg'] * (1 - df['bodyFatPercent'] / 100)
df['gripPerLeanMass'] = df['gripForce'] / lean_mass

# WHY: Grip force has DIFFERENT distributions for each gender. This interaction
# lets the model learn gender-specific grip thresholds.
df['genderXgrip'] = df['genderEncoded'] * df['gripForce']

feature_cols = ['age', 'heightCm', 'weightKg', 'bodyFatPercent', 'diastolic',
                'systolic', 'gripForce', 'sitAndBendForwardCm', 'sitUpsCounts',
                'broadJumpCm', 'genderEncoded',
                'bmi', 'sitUpsCounts_zGender', 'gripPerLeanMass', 'genderXgrip']

X = df[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Features: {len(feature_cols)} (11 original + 4 engineered)")

Features: 15 (11 original + 4 engineered)


### THREE TARGET CONFIGURATIONS


In [5]:
# 4-class: A, B, C, D (original)
le4 = LabelEncoder()
y4 = le4.fit_transform(df['performanceClass'])
names4 = le4.classes_

# 3-class: High (A), Average (B+C), Low (D)
# WHY: B and C overlap so heavily (<1 SD difference across all features)
# that they are statistically indistinguishable.
map3 = {'A': 'High', 'B': 'Average', 'C': 'Average', 'D': 'Low'}
df['class3'] = df['performanceClass'].map(map3)
le3 = LabelEncoder()
y3 = le3.fit_transform(df['class3'])
names3 = le3.classes_

# 2-class: Good (A+B), Poor (C+D)
# WHY: A binary split aligns with "above median" vs "below median" fitness.
map2 = {'A': 'Good', 'B': 'Good', 'C': 'Poor', 'D': 'Poor'}
df['class2'] = df['performanceClass'].map(map2)
le2 = LabelEncoder()
y2 = le2.fit_transform(df['class2'])
names2 = le2.classes_

print(f"\n4-class: {list(names4)} — {np.bincount(y4)}")
print(f"3-class: {list(names3)} — {np.bincount(y3)}")
print(f"2-class: {list(names2)} — {np.bincount(y2)}")


4-class: ['A', 'B', 'C', 'D'] — [3333 3338 3339 3262]
3-class: ['Average', 'High', 'Low'] — [6677 3333 3262]
2-class: ['Good', 'Poor'] — [6671 6601]


### Regression target (predict broadJumpCm instead — see explanation below)


In [6]:
# WHY broadJumpCm: Predicting gripForce would cause data leakage because
# gripPerLeanMass and genderXgrip are DERIVED from gripForce.
# BroadJump is a clean regression target with no leakage.
reg_feature_cols = [f for f in feature_cols if f != 'broadJumpCm']
X_reg = StandardScaler().fit_transform(df[reg_feature_cols].values)
y_reg = df['broadJumpCm'].values

### Primary 80:20 Split


In [7]:
targets = {'4-class': (y4, names4, le4),
           '3-class': (y3, names3, le3),
           '2-class': (y2, names2, le2)}

splits = {}
for tname, (y, names, le) in targets.items():
    Xtr, Xte, ytr, yte = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y)
    splits[tname] = (Xtr, Xte, ytr, yte, names)

## MODEL DEFINITIONS (used across all targets)


In [8]:
def get_models(optimal_k=7, optimal_depth=9):
    """Return dict of model instances."""
    return {
        'KNN': KNeighborsClassifier(n_neighbors=optimal_k),
        'Decision Tree': DecisionTreeClassifier(max_depth=optimal_depth, random_state=42),
        'SVM (Linear)': SVC(kernel='linear', C=1.0, random_state=42, probability=True),
        'SVM (RBF)': SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42, probability=True),
        'MLP (128,64)': MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                                       solver='adam', alpha=0.001, max_iter=500,
                                       random_state=42, early_stopping=True,
                                       validation_fraction=0.1, batch_size=128),
        'XGBoost': xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5,
                                      random_state=42, eval_metric='mlogloss', verbosity=0),
    }

## KNN HYPERPARAMETER TUNING


In [9]:
print("\n" + "=" * 70)
print("  KNN — FINDING OPTIMAL K")
print("=" * 70)
Xtr4, Xte4, ytr4, yte4, n4 = splits['4-class']
k_range = range(1, 31)
k_train, k_test = [], []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(Xtr4, ytr4)
    k_train.append(knn.score(Xtr4, ytr4))
    k_test.append(knn.score(Xte4, yte4))
optimal_k = k_range[np.argmax(k_test)]
print(f"  Optimal k = {optimal_k} (Test Acc = {max(k_test):.4f})")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, k_train, 'b-o', label='Train', markersize=3)
ax.plot(k_range, k_test, 'r-o', label='Test', markersize=3)
ax.axvline(optimal_k, color='green', linestyle='--', label=f'Best k={optimal_k}')
ax.set_xlabel('K'); ax.set_ylabel('Accuracy')
ax.set_title('KNN — Elbow Method for Optimal K', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
save(fig, 'KNN_Elbow')


  KNN — FINDING OPTIMAL K
  Optimal k = 29 (Test Acc = 0.6362)
  [Saved: plots/P26_KNN_Elbow.png]


'plots/P26_KNN_Elbow.png'

## DECISION TREE DEPTH TUNING


In [10]:
print("\n  DECISION TREE — FINDING OPTIMAL DEPTH")
d_range = range(1, 21)
d_train, d_test = [], []
for d in d_range:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(Xtr4, ytr4)
    d_train.append(dt.score(Xtr4, ytr4))
    d_test.append(dt.score(Xte4, yte4))
optimal_depth = d_range[np.argmax(d_test)]
print(f"  Optimal depth = {optimal_depth} (Test Acc = {max(d_test):.4f})")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(d_range, d_train, 'b-o', label='Train', markersize=4)
ax.plot(d_range, d_test, 'r-o', label='Test', markersize=4)
ax.axvline(optimal_depth, color='green', linestyle='--', label=f'Best depth={optimal_depth}')
ax.set_xlabel('Max Depth'); ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree — Depth Tuning', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
save(fig, 'DT_Depth_Tuning')


  DECISION TREE — FINDING OPTIMAL DEPTH
  Optimal depth = 10 (Test Acc = 0.7164)
  [Saved: plots/P27_DT_Depth_Tuning.png]


'plots/P27_DT_Depth_Tuning.png'

## TRAIN ALL MODELS ON ALL TARGETS (4-class, 3-class, 2-class)


In [11]:
print("\n" + "=" * 70)
print("  TRAINING ALL MODELS ON 4-CLASS, 3-CLASS, 2-CLASS")
print("=" * 70)
all_results = {}
all_preds = {}

for tname, (Xtr, Xte, ytr, yte, class_names) in splits.items():
    print(f"\n{'='*55}")
    print(f"  TARGET: {tname} ({list(class_names)})")
    print(f"{'='*55}")
    models = get_models(optimal_k, optimal_depth)
    results = {}
    preds = {}
    for mname, model in models.items():
        t0 = time.time()
        model.fit(Xtr, ytr)
        yp = model.predict(Xte)
        t1 = time.time() - t0
        acc = accuracy_score(yte, yp)
        prec = precision_score(yte, yp, average='weighted')
        rec = recall_score(yte, yp, average='weighted')
        f1 = f1_score(yte, yp, average='weighted')
        results[mname] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
                          'F1': f1, 'Time': t1}
        preds[mname] = (yp, model)
        print(f"  {mname:20s}  Acc={acc:.4f}  F1={f1:.4f}  ({t1:.1f}s)")
    all_results[tname] = results
    all_preds[tname] = preds


  TRAINING ALL MODELS ON 4-CLASS, 3-CLASS, 2-CLASS

  TARGET: 4-class (['A', 'B', 'C', 'D'])
  KNN                   Acc=0.6362  F1=0.6362  (0.4s)
  Decision Tree         Acc=0.7164  F1=0.7185  (0.1s)
  SVM (Linear)          Acc=0.6256  F1=0.6248  (20.2s)
  SVM (RBF)             Acc=0.7288  F1=0.7288  (21.0s)
  MLP (128,64)          Acc=0.7552  F1=0.7561  (4.2s)
  XGBoost               Acc=0.7744  F1=0.7744  (3.8s)

  TARGET: 3-class (['Average', 'High', 'Low'])
  KNN                   Acc=0.7578  F1=0.7555  (0.4s)
  Decision Tree         Acc=0.7891  F1=0.7901  (0.1s)
  SVM (Linear)          Acc=0.7416  F1=0.7405  (15.3s)
  SVM (RBF)             Acc=0.8060  F1=0.8064  (16.4s)
  MLP (128,64)          Acc=0.8147  F1=0.8154  (4.3s)
  XGBoost               Acc=0.8358  F1=0.8365  (2.8s)

  TARGET: 2-class (['Good', 'Poor'])
  KNN                   Acc=0.8403  F1=0.8391  (0.4s)
  Decision Tree         Acc=0.8618  F1=0.8614  (0.1s)
  SVM (Linear)          Acc=0.8234  F1=0.8233  (15.4s)
  SVM

### Comparison bar chart


In [12]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
colors = ['#2196F3', '#4CAF50', '#FF5722', '#E91E63', '#9C27B0', '#FF9800']
for ax, tname in zip(axes, ['4-class', '3-class', '2-class']):
    res = all_results[tname]
    model_names = list(res.keys())
    accs = [res[m]['Accuracy'] for m in model_names]
    bars = ax.bar(range(len(model_names)), accs, color=colors, alpha=0.85)
    for bar, a in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2, a+0.005, f'{a:.3f}', ha='center', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=25, ha='right', fontsize=8)
    ax.set_title(f'{tname}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Accuracy'); ax.set_ylim(0.5, 1.05)
fig.suptitle('All Models — Accuracy Across 4/3/2 Class Targets', fontsize=15, fontweight='bold')
plt.tight_layout()
save(fig, 'All_Models_Comparison')

  [Saved: plots/P28_All_Models_Comparison.png]


'plots/P28_All_Models_Comparison.png'

### Confusion matrices for 4-class (all models)


In [13]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (mname, (yp, _)) in zip(axes.flat, all_preds['4-class'].items()):
    cm = confusion_matrix(splits['4-class'][3], yp)
    ConfusionMatrixDisplay(cm, display_labels=names4).plot(ax=ax, cmap='Blues')
    acc = accuracy_score(splits['4-class'][3], yp)
    ax.set_title(f'{mname} (Acc={acc:.4f})', fontsize=11, fontweight='bold')
fig.suptitle('Confusion Matrices — 4-Class', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'ConfMatrix_4class')

  [Saved: plots/P29_ConfMatrix_4class.png]


'plots/P29_ConfMatrix_4class.png'

### Confusion matrices for 3-class


In [14]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (mname, (yp, _)) in zip(axes.flat, all_preds['3-class'].items()):
    cm = confusion_matrix(splits['3-class'][3], yp)
    ConfusionMatrixDisplay(cm, display_labels=names3).plot(ax=ax, cmap='Greens')
    acc = accuracy_score(splits['3-class'][3], yp)
    ax.set_title(f'{mname} (Acc={acc:.4f})', fontsize=11, fontweight='bold')
fig.suptitle('Confusion Matrices — 3-Class', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'ConfMatrix_3class')

  [Saved: plots/P30_ConfMatrix_3class.png]


'plots/P30_ConfMatrix_3class.png'

### Classification reports (4-class, printed)


In [15]:
print("\n" + "=" * 70)
print("  DETAILED CLASSIFICATION REPORTS (4-class)")
print("=" * 70)
for mname, (yp, _) in all_preds['4-class'].items():
    print(f"\n  {mname}:")
    print(classification_report(splits['4-class'][3], yp, target_names=names4))


  DETAILED CLASSIFICATION REPORTS (4-class)

  KNN:
              precision    recall  f1-score   support

           A       0.64      0.86      0.73       667
           B       0.49      0.49      0.49       668
           C       0.57      0.53      0.55       668
           D       0.93      0.66      0.77       652

    accuracy                           0.64      2655
   macro avg       0.66      0.64      0.64      2655
weighted avg       0.66      0.64      0.64      2655


  Decision Tree:
              precision    recall  f1-score   support

           A       0.70      0.83      0.76       667
           B       0.57      0.64      0.60       668
           C       0.74      0.61      0.67       668
           D       0.92      0.78      0.84       652

    accuracy                           0.72      2655
   macro avg       0.73      0.72      0.72      2655
weighted avg       0.73      0.72      0.72      2655


  SVM (Linear):
              precision    recall  f1-scor

## LINEAR REGRESSION (Predicting Grip Force)


In [16]:
print("\n" + "=" * 70)
print("  LINEAR REGRESSION — Predicting Grip Force")
print("=" * 70)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_models = {
    'Linear Regression': LinearRegression(),
    'KNN Regressor': KNeighborsRegressor(n_neighbors=10),
    'DT Regressor': DecisionTreeRegressor(max_depth=10, random_state=42),
    'SVR (RBF)': SVR(kernel='rbf', C=10.0, gamma='scale'),
    'MLP Regressor': MLPRegressor(hidden_layer_sizes=(128,64), activation='relu',
                                   solver='adam', alpha=0.01, max_iter=500,
                                   early_stopping=True, random_state=42),
}
reg_results = {}
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for (mname, model), ax in zip(reg_models.items(), axes.flat):
    model.fit(Xtr_r, ytr_r)
    yp = model.predict(Xte_r)
    r2 = r2_score(yte_r, yp)
    mae = mean_absolute_error(yte_r, yp)
    rmse = np.sqrt(mean_squared_error(yte_r, yp))
    reg_results[mname] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"  {mname:20s}  R²={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    ax.scatter(yte_r, yp, alpha=0.2, s=8, color='#42A5F5')
    ax.plot([yte_r.min(), yte_r.max()], [yte_r.min(), yte_r.max()], 'r--', lw=2)
    ax.set_title(f'{mname}\nR²={r2:.4f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
axes.flat[-1].set_visible(False)
fig.suptitle('Regression Models — Grip Force Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'Regression_All_Models')


  LINEAR REGRESSION — Predicting Grip Force
  Linear Regression     R²=0.8007  MAE=13.39  RMSE=17.79
  KNN Regressor         R²=0.7858  MAE=14.00  RMSE=18.44
  DT Regressor          R²=0.7418  MAE=15.33  RMSE=20.25
  SVR (RBF)             R²=0.8058  MAE=13.26  RMSE=17.56
  MLP Regressor         R²=0.8084  MAE=13.23  RMSE=17.45
  [Saved: plots/P31_Regression_All_Models.png]


'plots/P31_Regression_All_Models.png'

## MULTI-SPLIT EVALUATION (80:20, 70:30, 50:50)


In [17]:
print("\n" + "=" * 70)
print("  MULTI-SPLIT EVALUATION (80:20 / 70:30 / 50:50)")
print("=" * 70)
split_ratios = [0.2, 0.3, 0.5]
split_labels = ['80:20', '70:30', '50:50']
split_results = {tname: {m: [] for m in get_models(optimal_k, optimal_depth).keys()}
                 for tname in targets}

for tname, (y, names, le) in targets.items():
    for split_lbl, test_size in zip(split_labels, split_ratios):
        Xtr, Xte, ytr, yte = train_test_split(X_scaled, y, test_size=test_size,
                                               random_state=42, stratify=y)
        for mname, model in get_models(optimal_k, optimal_depth).items():
            m = clone(model)
            m.fit(Xtr, ytr)
            acc = m.score(Xte, yte)
            split_results[tname][mname].append(acc)
            if tname == '4-class':
                print(f"  {split_lbl} | {mname:20s}: {acc:.4f}")

# Plot multi-split for 4-class
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(split_labels))
width = 0.12
colors = ['#2196F3', '#4CAF50', '#FF5722', '#E91E63', '#9C27B0', '#FF9800']
model_names = list(get_models(optimal_k, optimal_depth).keys())
for i, (mname, c) in enumerate(zip(model_names, colors)):
    vals = split_results['4-class'][mname]
    ax.bar(x + i*width, vals, width, label=mname, color=c, alpha=0.85)
ax.set_xticks(x + width*2.5)
ax.set_xticklabels(split_labels)
ax.set_ylabel('Accuracy'); ax.set_ylim(0.4, 1.0)
ax.set_title('4-Class Accuracy Across Data Splits', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='lower left')
plt.tight_layout()
save(fig, 'MultiSplit_4class')


  MULTI-SPLIT EVALUATION (80:20 / 70:30 / 50:50)
  80:20 | KNN                 : 0.6362
  80:20 | Decision Tree       : 0.7164
  80:20 | SVM (Linear)        : 0.6256
  80:20 | SVM (RBF)           : 0.7288
  80:20 | MLP (128,64)        : 0.7552
  80:20 | XGBoost             : 0.7744
  70:30 | KNN                 : 0.6298
  70:30 | Decision Tree       : 0.7032
  70:30 | SVM (Linear)        : 0.6291
  70:30 | SVM (RBF)           : 0.7177
  70:30 | MLP (128,64)        : 0.7516
  70:30 | XGBoost             : 0.7687
  50:50 | KNN                 : 0.6207
  50:50 | Decision Tree       : 0.6787
  50:50 | SVM (Linear)        : 0.6243
  50:50 | SVM (RBF)           : 0.7054
  50:50 | MLP (128,64)        : 0.7294
  50:50 | XGBoost             : 0.7536
  [Saved: plots/P32_MultiSplit_4class.png]


'plots/P32_MultiSplit_4class.png'

## 5-FOLD CROSS-VALIDATION


In [18]:
print("\n" + "=" * 70)
print("  5-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for tname in ['4-class', '3-class', '2-class']:
    y_cv = targets[tname][0]
    cv_results[tname] = {}
    print(f"\n  --- {tname} ---")
    for mname, model in get_models(optimal_k, optimal_depth).items():
        fold_accs = []
        for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_cv), 1):
            m = clone(model)
            m.fit(X_scaled[train_idx], y_cv[train_idx])
            yp = m.predict(X_scaled[test_idx])
            fold_accs.append(accuracy_score(y_cv[test_idx], yp))
        mean_a, std_a = np.mean(fold_accs), np.std(fold_accs)
        cv_results[tname][mname] = {'mean': mean_a, 'std': std_a, 'folds': fold_accs}
        print(f"  {mname:20s}  Mean={mean_a:.4f} ± {std_a:.4f}  Folds: {[f'{a:.4f}' for a in fold_accs]}")

# CV bar chart with error bars
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, tname in zip(axes, ['4-class', '3-class', '2-class']):
    res = cv_results[tname]
    names_m = list(res.keys())
    means = [res[m]['mean'] for m in names_m]
    stds = [res[m]['std'] for m in names_m]
    bars = ax.bar(range(len(names_m)), means, yerr=stds, capsize=6, color=colors, alpha=0.85)
    for bar, m_val in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{m_val:.3f}',
                ha='center', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(names_m)))
    ax.set_xticklabels(names_m, rotation=25, ha='right', fontsize=8)
    ax.set_title(f'{tname} (5-Fold CV)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Mean Accuracy'); ax.set_ylim(0.5, 1.05)
fig.suptitle('5-Fold Cross-Validation Results', fontsize=15, fontweight='bold')
plt.tight_layout()
save(fig, 'CV_5Fold_All')

# Per-fold stability
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, tname in zip(axes, ['4-class', '3-class', '2-class']):
    for mname, c in zip(model_names, colors):
        ax.plot(range(1,6), cv_results[tname][mname]['folds'], 'o-', color=c, label=mname, lw=2)
    ax.set_xlabel('Fold'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{tname} — Per-Fold Stability', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7); ax.set_xticks(range(1,6))
plt.tight_layout()
save(fig, 'CV_PerFold_Stability')


  5-FOLD STRATIFIED CROSS-VALIDATION

  --- 4-class ---
  KNN                   Mean=0.6261 ± 0.0105  Folds: ['0.6301', '0.6094', '0.6274', '0.6221', '0.6417']
  Decision Tree         Mean=0.7023 ± 0.0068  Folds: ['0.7021', '0.6908', '0.7012', '0.7114', '0.7061']
  SVM (Linear)          Mean=0.6283 ± 0.0103  Folds: ['0.6324', '0.6181', '0.6209', '0.6236', '0.6466']
  SVM (RBF)             Mean=0.7119 ± 0.0045  Folds: ['0.7119', '0.7126', '0.7102', '0.7054', '0.7193']
  MLP (128,64)          Mean=0.7330 ± 0.0052  Folds: ['0.7341', '0.7322', '0.7280', '0.7283', '0.7423']
  XGBoost               Mean=0.7625 ± 0.0100  Folds: ['0.7684', '0.7718', '0.7558', '0.7460', '0.7705']

  --- 3-class ---
  KNN                   Mean=0.7529 ± 0.0095  Folds: ['0.7548', '0.7405', '0.7611', '0.7434', '0.7645']
  Decision Tree         Mean=0.7804 ± 0.0056  Folds: ['0.7812', '0.7714', '0.7781', '0.7833', '0.7882']
  SVM (Linear)          Mean=0.7495 ± 0.0055  Folds: ['0.7492', '0.7409', '0.7551', '0.7468'

'plots/P34_CV_PerFold_Stability.png'

## FEATURE IMPORTANCE (DT + XGBoost)


In [19]:
print("\n" + "=" * 70)
print("  FEATURE IMPORTANCE")
print("=" * 70)
Xtr4, Xte4, ytr4, yte4, n4 = splits['4-class']
dt4 = DecisionTreeClassifier(max_depth=optimal_depth, random_state=42)
dt4.fit(Xtr4, ytr4)
xgb4 = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                           random_state=42, eval_metric='mlogloss', verbosity=0)
xgb4.fit(Xtr4, ytr4)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (model, title) in zip(axes, [(dt4, 'Decision Tree'), (xgb4, 'XGBoost')]):
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
    c_list = ['#F44336' if v>0.1 else '#FF9800' if v>0.05 else '#2196F3' for v in imp.values]
    imp.plot(kind='barh', ax=ax, color=c_list)
    ax.set_title(f'{title} Feature Importance', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
fig.suptitle('Feature Importance — 4-Class', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'Feature_Importance')


  FEATURE IMPORTANCE
  [Saved: plots/P35_Feature_Importance.png]


'plots/P35_Feature_Importance.png'

## SHAP VALUES (XGBoost)


In [20]:
print("\n  Computing SHAP values...")
try:
    import shap
    explainer = shap.TreeExplainer(xgb4)
    shap_values = explainer.shap_values(Xte4[:500])
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, Xte4[:500], feature_names=feature_cols, show=False,
                      max_display=15, plot_size=None)
    plt.title('SHAP Feature Importance (XGBoost, 4-Class)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    save(plt.gcf(), 'SHAP_Summary')
except Exception as e:
    print(f"  SHAP failed: {e}")


  Computing SHAP values...
  SHAP failed: No module named 'shap'


## LEARNING CURVES


In [21]:
print("\n  Computing Learning Curves...")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
lc_models = {
    'KNN': KNeighborsClassifier(n_neighbors=optimal_k),
    'Decision Tree': DecisionTreeClassifier(max_depth=optimal_depth, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(128,64), max_iter=300, random_state=42, early_stopping=True),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, verbosity=0, eval_metric='mlogloss'),
}
train_sizes = np.linspace(0.1, 1.0, 8)

for ax, (mname, model) in zip(axes.flat, lc_models.items()):
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X_scaled, y4, train_sizes=train_sizes, cv=3, scoring='accuracy',
        n_jobs=-1, random_state=42)
    tr_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    tr_std = train_scores.std(axis=1)
    val_std = val_scores.std(axis=1)
    ax.fill_between(train_sizes_abs, tr_mean-tr_std, tr_mean+tr_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes_abs, val_mean-val_std, val_mean+val_std, alpha=0.1, color='red')
    ax.plot(train_sizes_abs, tr_mean, 'b-o', label='Train', markersize=4)
    ax.plot(train_sizes_abs, val_mean, 'r-o', label='Validation', markersize=4)
    ax.set_title(mname, fontweight='bold'); ax.set_xlabel('Training Size')
    ax.set_ylabel('Accuracy'); ax.legend(fontsize=8)
axes.flat[-1].set_visible(False)
fig.suptitle('Learning Curves — 4-Class (Train vs Validation)', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'Learning_Curves')


  Computing Learning Curves...
  [Saved: plots/P36_Learning_Curves.png]


'plots/P36_Learning_Curves.png'

## ROC CURVES (One-vs-Rest for 4-class)


In [22]:
print("\n  Computing ROC Curves...")
Xtr4, Xte4, ytr4, yte4, n4 = splits['4-class']
y_bin = label_binarize(yte4, classes=range(len(names4)))
n_classes = y_bin.shape[1]

roc_models = {
    'XGBoost': xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                  random_state=42, eval_metric='mlogloss', verbosity=0),
    'MLP': MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500, random_state=42,
                          early_stopping=True, batch_size=128),
    'SVM (RBF)': SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42, probability=True),
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (mname, model) in zip(axes, roc_models.items()):
    model.fit(Xtr4, ytr4)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(Xte4)
    else:
        y_score = model.decision_function(Xte4)
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f'Class {names4[i]} (AUC={roc_auc:.3f})')
    ax.plot([0,1], [0,1], 'k--', lw=1)
    ax.set_title(f'{mname}', fontweight='bold')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)
fig.suptitle('ROC Curves (One-vs-Rest) — 4-Class', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'ROC_Curves_4class')


  Computing ROC Curves...
  [Saved: plots/P37_ROC_Curves_4class.png]


'plots/P37_ROC_Curves_4class.png'

## SCALING IMPACT ANALYSIS


In [23]:
print("\n" + "=" * 70)
print("  FEATURE SCALING IMPACT — Scaled vs Unscaled")
print("=" * 70)
Xtr_raw, Xte_raw, _, _ = train_test_split(X, y4, test_size=0.2, random_state=42, stratify=y4)
scale_models = {
    'KNN': KNeighborsClassifier(n_neighbors=optimal_k),
    'SVM (RBF)': SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=optimal_depth, random_state=42),
}
scale_res = []
for mname, model in scale_models.items():
    # Unscaled
    m1 = clone(model); m1.fit(Xtr_raw, ytr4)
    acc_raw = m1.score(Xte_raw, yte4)
    # Scaled
    m2 = clone(model); m2.fit(Xtr4, ytr4)
    acc_scaled = m2.score(Xte4, yte4)
    diff = acc_scaled - acc_raw
    scale_res.append({'Model': mname, 'Unscaled': acc_raw, 'Scaled': acc_scaled, 'Diff': diff})
    print(f"  {mname:20s}  Unscaled={acc_raw:.4f}  Scaled={acc_scaled:.4f}  Δ={diff:+.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
sr = pd.DataFrame(scale_res)
x = np.arange(len(sr))
ax.bar(x-0.15, sr['Unscaled'], 0.3, label='Unscaled', color='#FF9800', alpha=0.85)
ax.bar(x+0.15, sr['Scaled'], 0.3, label='Scaled', color='#4CAF50', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(sr['Model'])
ax.set_ylabel('Accuracy'); ax.set_title('Impact of Feature Scaling', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
save(fig, 'Scaling_Impact')


  FEATURE SCALING IMPACT — Scaled vs Unscaled
  KNN                   Unscaled=0.5789  Scaled=0.6362  Δ=+0.0573
  SVM (RBF)             Unscaled=0.6414  Scaled=0.7288  Δ=+0.0874
  Decision Tree         Unscaled=0.7164  Scaled=0.7164  Δ=+0.0000
  [Saved: plots/P38_Scaling_Impact.png]


'plots/P38_Scaling_Impact.png'

## PER-CLASS PRECISION/RECALL ANALYSIS


In [24]:
print("\n  Per-Class Analysis (4-class, XGBoost):")
yp_xgb = all_preds['4-class']['XGBoost'][0]
yte_4 = splits['4-class'][3]
cr = classification_report(yte_4, yp_xgb, target_names=names4, output_dict=True)
for cls in names4:
    m = cr[cls]
    print(f"  Class {cls}: Precision={m['precision']:.4f}  Recall={m['recall']:.4f}  F1={m['f1-score']:.4f}")
print("  Observation: Classes B and C have the lowest F1 scores due to their overlapping distributions.")


  Per-Class Analysis (4-class, XGBoost):
  Class A: Precision=0.7564  Recall=0.8801  F1=0.8136
  Class B: Precision=0.6647  Recall=0.6647  F1=0.6647
  Class C: Precision=0.7691  Recall=0.7081  F1=0.7373
  Class D: Precision=0.9262  Recall=0.8466  F1=0.8846
  Observation: Classes B and C have the lowest F1 scores due to their overlapping distributions.


## HYPERPARAMETER SENSITIVITY (SVM C-gamma heatmap)


In [25]:
print("\n  SVM C-Gamma Sensitivity Analysis...")
C_vals = [0.1, 1, 10, 100]
gamma_vals = [0.001, 0.01, 0.1, 1]
svm_grid = np.zeros((len(C_vals), len(gamma_vals)))
for i, C in enumerate(C_vals):
    for j, g in enumerate(gamma_vals):
        svm = SVC(kernel='rbf', C=C, gamma=g, random_state=42)
        svm.fit(Xtr4, ytr4)
        svm_grid[i, j] = svm.score(Xte4, yte4)
        print(f"    C={C:6.1f}  gamma={g:5.3f}  Acc={svm_grid[i,j]:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(svm_grid, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=gamma_vals, yticklabels=C_vals, ax=ax)
ax.set_xlabel('Gamma'); ax.set_ylabel('C')
ax.set_title('SVM (RBF) — C vs Gamma Sensitivity Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
save(fig, 'SVM_CGamma_Heatmap')


  SVM C-Gamma Sensitivity Analysis...
    C=   0.1  gamma=0.001  Acc=0.5382
    C=   0.1  gamma=0.010  Acc=0.6226
    C=   0.1  gamma=0.100  Acc=0.6652
    C=   0.1  gamma=1.000  Acc=0.4218
    C=   1.0  gamma=0.001  Acc=0.6196
    C=   1.0  gamma=0.010  Acc=0.6561
    C=   1.0  gamma=0.100  Acc=0.7104
    C=   1.0  gamma=1.000  Acc=0.6437
    C=  10.0  gamma=0.001  Acc=0.6320
    C=  10.0  gamma=0.010  Acc=0.6976
    C=  10.0  gamma=0.100  Acc=0.7254
    C=  10.0  gamma=1.000  Acc=0.6256
    C= 100.0  gamma=0.001  Acc=0.6535
    C= 100.0  gamma=0.010  Acc=0.7247
    C= 100.0  gamma=0.100  Acc=0.6814
    C= 100.0  gamma=1.000  Acc=0.6256
  [Saved: plots/P39_SVM_CGamma_Heatmap.png]


'plots/P39_SVM_CGamma_Heatmap.png'

## VOTING ENSEMBLE (G24)


In [26]:
print("\n" + "=" * 70)
print("  VOTING ENSEMBLE — Combining Required Models")
print("=" * 70)
voting = VotingClassifier(
    estimators=[
        ('knn', KNeighborsClassifier(n_neighbors=optimal_k)),
        ('dt', DecisionTreeClassifier(max_depth=optimal_depth, random_state=42)),
        ('svm', SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42, probability=True)),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500, random_state=42,
                              early_stopping=True, batch_size=128)),
    ],
    voting='soft'
)
voting.fit(Xtr4, ytr4)
yp_vote = voting.predict(Xte4)
acc_vote = accuracy_score(yte4, yp_vote)
print(f"  Voting Ensemble Accuracy (4-class): {acc_vote:.4f}")
print(classification_report(yte4, yp_vote, target_names=names4))

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(yte4, yp_vote)
ConfusionMatrixDisplay(cm, display_labels=names4).plot(ax=ax, cmap='Purples')
ax.set_title(f'Voting Ensemble (Acc={acc_vote:.4f})', fontsize=13, fontweight='bold')
plt.tight_layout()
save(fig, 'Voting_Ensemble_CM')


  VOTING ENSEMBLE — Combining Required Models
  Voting Ensemble Accuracy (4-class): 0.7578
              precision    recall  f1-score   support

           A       0.75      0.86      0.80       667
           B       0.63      0.69      0.66       668
           C       0.76      0.68      0.71       668
           D       0.93      0.80      0.86       652

    accuracy                           0.76      2655
   macro avg       0.77      0.76      0.76      2655
weighted avg       0.77      0.76      0.76      2655

  [Saved: plots/P40_Voting_Ensemble_CM.png]


'plots/P40_Voting_Ensemble_CM.png'

## STATISTICAL SIGNIFICANCE (McNemar's Test Approximation)


In [27]:
print("\n" + "=" * 70)
print("  STATISTICAL SIGNIFICANCE — Pairwise Model Comparison")
print("=" * 70)
print("  Using 5-Fold CV paired t-test to compare models:")
from scipy.stats import ttest_rel
best_model_name = max(cv_results['4-class'], key=lambda m: cv_results['4-class'][m]['mean'])
best_folds = cv_results['4-class'][best_model_name]['folds']
print(f"  Best model: {best_model_name} (mean CV acc = {np.mean(best_folds):.4f})")
for mname in cv_results['4-class']:
    if mname == best_model_name:
        continue
    other_folds = cv_results['4-class'][mname]['folds']
    t_stat, p_val = ttest_rel(best_folds, other_folds)
    sig = "YES (p<0.05)" if p_val < 0.05 else "NO (p>=0.05)"
    print(f"  {best_model_name} vs {mname}: t={t_stat:.3f}, p={p_val:.4f} → Significant: {sig}")


  STATISTICAL SIGNIFICANCE — Pairwise Model Comparison
  Using 5-Fold CV paired t-test to compare models:
  Best model: XGBoost (mean CV acc = 0.7625)
  XGBoost vs KNN: t=19.786, p=0.0000 → Significant: YES (p<0.05)
  XGBoost vs Decision Tree: t=7.871, p=0.0014 → Significant: YES (p<0.05)
  XGBoost vs SVM (Linear): t=23.996, p=0.0000 → Significant: YES (p<0.05)
  XGBoost vs SVM (RBF): t=14.891, p=0.0001 → Significant: YES (p<0.05)
  XGBoost vs MLP (128,64): t=8.084, p=0.0013 → Significant: YES (p<0.05)


## MLP ARCHITECTURE COMPARISON (G5)


In [28]:
print("\n  MLP Architecture Comparison:")
archs = [(64,), (128,), (128, 64), (256, 128)]
for arch in archs:
    mlp = MLPClassifier(hidden_layer_sizes=arch, max_iter=500, random_state=42,
                        early_stopping=True, batch_size=128)
    scores = cross_val_score(mlp, X_scaled, y4, cv=3, scoring='accuracy')
    print(f"  MLP{arch}: CV Acc = {scores.mean():.4f} ± {scores.std():.4f}")


  MLP Architecture Comparison:
  MLP(64,): CV Acc = 0.7198 ± 0.0087
  MLP(128,): CV Acc = 0.7224 ± 0.0066
  MLP(128, 64): CV Acc = 0.7316 ± 0.0099
  MLP(256, 128): CV Acc = 0.7306 ± 0.0057


## FINAL RESULTS SUMMARY


In [29]:
print("\n" + "=" * 70)
print("  ═══ FINAL RESULTS SUMMARY ═══")
print("=" * 70)

# Comprehensive results table
print(f"\n{'Model':<20} | {'4-class':>8} | {'3-class':>8} | {'2-class':>8}")
print(f"{'-'*52}")
for mname in get_models(optimal_k, optimal_depth).keys():
    a4 = all_results['4-class'][mname]['Accuracy']
    a3 = all_results['3-class'][mname]['Accuracy']
    a2 = all_results['2-class'][mname]['Accuracy']
    print(f"{mname:<20} | {a4:8.4f} | {a3:8.4f} | {a2:8.4f}")
print(f"{'Voting Ensemble':<20} | {acc_vote:8.4f} |    —     |    —    ")

print(f"\n  Regression (Grip Force):")
print(f"  {'Model':<20} | {'R²':>6} | {'MAE':>8} | {'RMSE':>8}")
for mname, r in reg_results.items():
    print(f"  {mname:<20} | {r['R2']:6.4f} | {r['MAE']:8.2f} | {r['RMSE']:8.2f}")

# Final comparison chart
fig, ax = plt.subplots(figsize=(12, 7))
all_model_names = list(get_models(optimal_k, optimal_depth).keys()) + ['Voting Ensemble']
data = []
for mname in all_model_names[:-1]:
    data.append([all_results[t][mname]['Accuracy'] for t in ['4-class','3-class','2-class']])
data.append([acc_vote, None, None])
x = np.arange(len(all_model_names))
width = 0.25
colors_3 = ['#F44336', '#4CAF50', '#2196F3']
for i, (tname, c) in enumerate(zip(['4-class','3-class','2-class'], colors_3)):
    vals = [d[i] if d[i] is not None else 0 for d in data]
    ax.bar(x + i*width, vals, width, label=tname, color=c, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(all_model_names, rotation=20, ha='right')
ax.set_ylabel('Accuracy'); ax.set_ylim(0.5, 1.05)
ax.set_title('Final Model Comparison — 4/3/2 Class Targets', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
save(fig, 'Final_Comparison_All')

print(f"\n{'='*70}")
print(f"  ✅ PART 2 COMPLETE — {plot_num[0]-25} plots generated")
print(f"  ✅ Total plots in folder: {plot_num[0]}")
print(f"{'='*70}")


  ═══ FINAL RESULTS SUMMARY ═══

Model                |  4-class |  3-class |  2-class
----------------------------------------------------
KNN                  |   0.6362 |   0.7578 |   0.8403
Decision Tree        |   0.7164 |   0.7891 |   0.8618
SVM (Linear)         |   0.6256 |   0.7416 |   0.8234
SVM (RBF)            |   0.7288 |   0.8060 |   0.8648
MLP (128,64)         |   0.7552 |   0.8147 |   0.8719
XGBoost              |   0.7744 |   0.8358 |   0.8866
Voting Ensemble      |   0.7578 |    —     |    —    

  Regression (Grip Force):
  Model                |     R² |      MAE |     RMSE
  Linear Regression    | 0.8007 |    13.39 |    17.79
  KNN Regressor        | 0.7858 |    14.00 |    18.44
  DT Regressor         | 0.7418 |    15.33 |    20.25
  SVR (RBF)            | 0.8058 |    13.26 |    17.56
  MLP Regressor        | 0.8084 |    13.23 |    17.45
  [Saved: plots/P41_Final_Comparison_All.png]

  ✅ PART 2 COMPLETE — 16 plots generated
  ✅ Total plots in folder: 41
